# <font color='green'>Fine Tunning Open Source LLM for Chatbot Entity List Assistant</font>

## Packages

In [1]:
!pip install -q bitsandbytes datasets accelerate loralib evaluate

In [2]:
!pip install -q git+https://github.com/huggingface/transformers.git@main git+https://github.com/huggingface/peft.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


## Imports

In [4]:
import os
import json
import torch
import evaluate
import torch.nn as nn
import transformers
import bitsandbytes as bnb
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM
from transformers import BitsAndBytesConfig, pipeline
from peft import LoraConfig, get_peft_model
from datasets import Dataset, Features, ClassLabel, Value, Sequence
import warnings
warnings.filterwarnings('ignore')

## Check GPU

In [5]:
if torch.cuda.is_available():
  print('Nº of GPUs:', torch.cuda.device_count())
  print('GPU Name:', torch.cuda.get_device_name(0))
  print('GPU Memory:', torch.cuda.get_device_properties(0).total_memory / 1e9)

Nº of GPUs: 1
GPU Name: NVIDIA A100-SXM4-40GB
GPU Memory: 42.405855232


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Quantization Parameters

In [7]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_use_double_quant = True,
    llm_int8_enable_fp32_cpu_offload = True
    )

# Load Tokenizer and Model

In [8]:
model = AutoModelForCausalLM.from_pretrained(
    "tiiuae/falcon-7b",
    quantization_config = quantization_config,
    device_map = 'auto'
  )

[transformers] You are loading your model using eetq but no linear modules were found in your model. Please double check your model architecture, or submit an issue on github if you think this is a bug.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

In [9]:
tokenizer = AutoTokenizer.from_pretrained("tiiuae/falcon-7b")

## Freeze Original Weights

In [10]:
# Loop
for param in model.parameters():
  param.requires_grad = False
  if param.ndim == 1:
    param.data = param.data.to(torch.float32)

## Gradient Checkpoint

In [11]:
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

## Tensor Conversion

In [12]:
class CastOutputFloat(nn.Sequential):
  def forward(self, x):
    return super().forward(x).to(torch.float32)

model.lm_head = CastOutputFloat(model.lm_head)

## Fine Tuning Parameters

In [13]:
# LoRa Config
config = LoraConfig(
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM"
)

In [14]:
model = get_peft_model(model, config)

In [15]:
# Print training parameters
def print_trainable_parameters(model):
  trainable_params = 0
  all_param = 0
  for _, param in model.named_parameters():
    all_param += param.numel()
    if param.requires_grad:
      trainable_params += param.numel()
  print(f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}")

In [16]:
print_trainable_parameters(model)

trainable params: 4718592 || all params: 6926439296 || trainable%: 0.06812435363037071


## Load Data

In [17]:
file_1 = open("/content/drive/Othercomputers/Meu laptop/Agentic_AI/LLM_PLN/cap_10/dataset_1.json")
data_1 = json.load(file_1)

In [18]:
data_1

{'questions': [{'question': 'What is the Entity List?',
   'answer': 'The Bureau of Industry and Security (BIS) publishes the names of certain foreign entities including businesses, research institutions, government and private organizations, individuals, and other types of legal persons—that are subject to specific license requirements for the export, reexport, and transfer (in-country) of specified items. These entities comprise the Entity List, which is found at Supplement no. 4 to part 744 of the Export Administration Regulations (EAR). The entities on the Entity List are subject to individual licensing requirements and policies supplemental to those found elsewhere in the EAR. Certain non-listed foreign affiliates of listed entities are also subject to the Entity List license requirements and other requirements because they meet the Affiliates Rule criteria.'},
  {'question': 'What is the background and purpose of the Entity List?',
   'answer': 'BIS first published the Entity Lis

In [19]:
# List for question and answer
questions = []
answers = []

In [20]:
for i in data_1["questions"]:
  questions += [i["question"]]
  answers += [i["answer"]]

In [22]:
# Original Dataset
data_1["questions"][0]

{'question': 'What is the Entity List?',
 'answer': 'The Bureau of Industry and Security (BIS) publishes the names of certain foreign entities including businesses, research institutions, government and private organizations, individuals, and other types of legal persons—that are subject to specific license requirements for the export, reexport, and transfer (in-country) of specified items. These entities comprise the Entity List, which is found at Supplement no. 4 to part 744 of the Export Administration Regulations (EAR). The entities on the Entity List are subject to individual licensing requirements and policies supplemental to those found elsewhere in the EAR. Certain non-listed foreign affiliates of listed entities are also subject to the Entity List license requirements and other requirements because they meet the Affiliates Rule criteria.'}

In [23]:
# First Question
questions[0]

'What is the Entity List?'

In [25]:
# Format data for model training
dataset = Dataset.from_dict({
    "id": list(range(len(questions))),
    "questions": questions, # Changed from 'question' to 'questions'
    "answers": answers      # Changed from 'answer' to 'answers'
    },
    features = Features({
      "id": Value(dtype = 'string'),
      "questions": Value(dtype = 'string'),
      "answers": Value(dtype = 'string')
    })
)

In [26]:
# Divide data into train and test
dataset = dataset.train_test_split(test_size = 0.15)

In [27]:
# Merge questions and answers
def merge_columns(register):
  register["output"] = register["questions"] + " ->: " + register["answers"]
  return register

In [28]:
train_dataset = dataset.map(merge_columns)

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

In [29]:
# Show Format
train_dataset["train"]["output"][0]

'Does the adoption of the Affiliates Rule mean that the Consolidated Screening List (CSL) will no longer be considered an exhaustive listing of foreign entities subject to Entity List requirements? ->: Yes. The adoption of the Affiliates Rule will mean that the Consolidated Screening List (CSL) will no longer comprise an exhaustive listing of foreign entities subject to Entity List license requirements. The CSL, for purposes of the Entity List, will only include the entities listed on the Entity List and will not reflect these additional foreign affiliates of listed entities that are owned 50 percent or more by one or more listed entities. Persons should screen proposed transactions against the CSL and separately screen for purposes of the Affiliates Rule.'

In [30]:
train_dataset["train"][0]

{'id': '44',
 'questions': 'Does the adoption of the Affiliates Rule mean that the Consolidated Screening List (CSL) will no longer be considered an exhaustive listing of foreign entities subject to Entity List requirements?',
 'answers': 'Yes. The adoption of the Affiliates Rule will mean that the Consolidated Screening List (CSL) will no longer comprise an exhaustive listing of foreign entities subject to Entity List license requirements. The CSL, for purposes of the Entity List, will only include the entities listed on the Entity List and will not reflect these additional foreign affiliates of listed entities that are owned 50 percent or more by one or more listed entities. Persons should screen proposed transactions against the CSL and separately screen for purposes of the Affiliates Rule.',
 'output': 'Does the adoption of the Affiliates Rule mean that the Consolidated Screening List (CSL) will no longer be considered an exhaustive listing of foreign entities subject to Entity Lis

In [31]:
# Tokenizing data
train_dataset = train_dataset.map(lambda samples: tokenizer(samples['output']), batched = True)

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

In [32]:
# Tokenized data
train_dataset["train"][0]

{'id': '44',
 'questions': 'Does the adoption of the Affiliates Rule mean that the Consolidated Screening List (CSL) will no longer be considered an exhaustive listing of foreign entities subject to Entity List requirements?',
 'answers': 'Yes. The adoption of the Affiliates Rule will mean that the Consolidated Screening List (CSL) will no longer comprise an exhaustive listing of foreign entities subject to Entity List license requirements. The CSL, for purposes of the Entity List, will only include the entities listed on the Entity List and will not reflect these additional foreign affiliates of listed entities that are owned 50 percent or more by one or more listed entities. Persons should screen proposed transactions against the CSL and separately screen for purposes of the Affiliates Rule.',
 'output': 'Does the adoption of the Affiliates Rule mean that the Consolidated Screening List (CSL) will no longer be considered an exhaustive listing of foreign entities subject to Entity Lis

## Setting training args

In [33]:
if tokenizer.pad_token == None:
  tokenizer.pad_token = tokenizer.eos_token

In [34]:
model_trainer = transformers.Trainer(
    model = model,
    train_dataset = train_dataset["train"],
    eval_dataset = train_dataset["test"],
    args = transformers.TrainingArguments(
        eval_strategy = "epoch",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 2,
        num_train_epochs = 10,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 1,
        output_dir = 'outputs',
        report_to = "none"
    ),
    data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm = False)
)

## Training the model

In [35]:
model.config.use_cache = False
model_trainer.train()

Epoch,Training Loss,Validation Loss
1,2.216848,2.165242
2,1.779711,1.957180
3,1.529908,1.830419
4,1.689130,1.738881
5,1.274314,1.710689
6,1.326734,1.671405
7,1.376571,1.688555
8,1.043490,1.662686
9,0.865609,1.679825
10,1.013543,1.680135


TrainOutput(global_step=110, training_loss=1.4545146833766591, metrics={'train_runtime': 74.8856, 'train_samples_per_second': 5.876, 'train_steps_per_second': 1.469, 'total_flos': 4197324590277120.0, 'train_loss': 1.4545146833766591, 'epoch': 10.0})

## Evaluating Model Performance

In [36]:
def model_predict(question):
  model.eval()
  device = next(model.parameters()).device
  batch = tokenizer(
      f"{question} ->: ",
      return_tensors = 'pt',
      padding = True,
      truncation = True
      )
  batch = {k: v.to(device) for k, v in batch.items()}
  with torch.no_grad(), torch.cuda.amp.autocast():
    output_tokens = model.generate(
        **batch,
        max_new_tokens = 50,
        pad_token_id = tokenizer.eos_token_id
    )
  return tokenizer.decode(output_tokens[0], skip_special_tokens = True)

In [37]:
# List of predictions
predictions = []

In [38]:
for i in train_dataset["test"]["questions"]:
  predictions.append(model_predict(i))

In [39]:
# Bleu method
bleu = evaluate.load('bleu')

In [40]:
#Extract real data
real_data = train_dataset["test"]["output"]

In [41]:
results = bleu.compute(predictions = predictions, references = real_data)

In [42]:
results

{'bleu': 0.2540077777378535,
 'precisions': [0.6639344262295082,
  0.473421926910299,
  0.4158249158249158,
  0.3873720136518771],
 'brevity_penalty': 0.5354808429158032,
 'length_ratio': 0.615539858728557,
 'translation_length': 610,
 'reference_length': 991}

## Deploy of the model

In [44]:
# Device
device = next(model.parameters()).device

In [45]:
question_1 = 'How can I see if a company is in the Entity List?'

In [46]:
tokenized_question = tokenizer(question_1, return_tensors = "pt", padding = True, truncation = True)

In [47]:
tokenized_question = {name: tensor.to(device) for name, tensor in tokenized_question.items()}

In [48]:
tokenized_question

{'input_ids': tensor([[ 1830,   418,   295,   760,   565,   241,  1438,   304,   272,   248,
          33704,  4702,    42]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [49]:
# Generate answer
with torch.no_grad(), torch.cuda.amp.autocast():
  predict_tokens = model.generate(**tokenized_question, max_new_tokens = 100, pad_token_id = tokenizer.eos_token_id)

In [50]:
# Decode anwser
tokenizer.decode(predict_tokens[0], skip_special_tokens = True)

'How can I see if a company is in the Entity List? ->: You can search the Entity List at to see if a company is listed on the Entity List. If a company is listed on the Entity List, you will need to obtain a license from the appropriate Bureau of the Department of Commerce to conduct transactions with that company. You can also search the Entity List at to see if a specific entity is listed on the Entity List. If a specific entity is listed on the Entity List, you will need to obtain a license from the appropriate Bureau of the'

In [51]:
# Saving Model
torch.save(model.state_dict(), "/content/drive/MyDrive/Modelos_LLMs/Fine_Tuning_Falcon_Entity_List/trained_model.pt")